# ARC-v0.29 — FEVER-E5 Prospective Operator/Horizon Replication

**Primary purpose:** replicate the NQ-GTE long-horizon recursive sign reversal on the existing FEVER-E5 severity-matched design.

This study is **prospective for the new FEVER-E5 operator/horizon outcomes**, but it is **not a pristine independent test** because FEVER-E5 short-horizon validation outcomes were already observed in ARC-v0.18/v0.25.

### Frozen primary
- Representation: IVF-PQ32 @ nprobe=64 → IVF-SQ8 @ nprobe=64
- Search effort: IVF-SQ8 @ nprobe=8 → nprobe=64
- Operators: anchored and recursive
- Structural long-horizon subset: 8 policies (mean-k20 and softmax-k20-τ0.1 for α∈{0.1,0.3,0.5,0.7})
- Prefixes: H ∈ {4,8,12,16,24,32,40,50}
- Primary: **recursive H=50 representation-minus-search H3abs**
- Replication success: **10,000 paired-query bootstrap 95% CI strictly below 0**

**Non-negotiable rule:** null, positive, negative, saturation, a different crossing point, or no crossing are all retained. Do not retune nprobe, policies, horizons, endpoints, split, or primary direction after validation begins.

In [ ]:
# Cell 1 — Imports, Drive mount, and immutable v0.29 protocol
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import gc, hashlib, json, math, os, shutil, time, warnings

import numpy as np
import pandas as pd

try:
    import faiss
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu", "pyarrow", "tqdm"])
    import faiss

try:
    from tqdm.auto import tqdm
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tqdm"])
    from tqdm.auto import tqdm

try:
    from google.colab import drive
except ImportError:
    drive = None

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 20260826
np.random.seed(SEED)

DIM = 384
N_DOCS = 5_416_568
N_FIT = 3_350
N_VAL = 3_316
TOP_RETRIEVE = 100
UTILITY_K = 10
HIGH_NPROBE = 64
MATCHED_NPROBE = 8
HORIZONS = [4, 8, 12, 16, 24, 32, 40, 50]
MAX_H = 50
BOOTSTRAP_REPS = 10_000
SEARCH_BATCH = 512
FEEDBACK_BATCH = 128
LOCALIZE_TO_SSD = True

EXPECTED_V025_PROTOCOL_SHA = "62351de5e3d2993129fd0b4f67e3fd49e06b97059b050cd800645d2d6c467e24"
EXPECTED_FIT_SHA = "85c01de943636a080abe10cb18cf5538b705b89f6f042a74c92bac366d89612c"
EXPECTED_VAL_SHA = "5e7bd8e3e5e3f0120b1e93726a19183285624c405ef68b34005c8766a9568b42"

V029_PROTOCOL_JSON = '{"classification": "Prospective with respect to new FEVER-E5 operator/horizon outcomes, but post-v0.25 with respect to the already observed FEVER-E5 short-horizon validation evidence. This is a prospective replication/extension, not a pristine independent test.", "created_at_utc": "2026-08-26T02:54:18.821143+00:00", "dataset": "BEIR FEVER", "dimension": 384, "encoder": "intfloat/e5-small-v2", "endpoints": {"H1": "OLS slope of paired query-state cosine distance over prefix 0..H", "H3abs": "OLS slope of |nDCG@10_high - nDCG@10_low| over prefix 0..H", "H3signed": "OLS slope of nDCG@10_high - nDCG@10_low over prefix 0..H", "primary": "recursive H=50 query-averaged H3abs representation minus severity-matched search effort", "secondary_metrics": ["MRR@10 H3abs", "Recall@10 H3abs", "final-minus-initial absolute utility gap"]}, "execution_safeguards": {"abort_on_backend_semantic_mismatch": true, "backend_audit_fit_queries": 256, "checkpoint_each_mechanism_operator_policy": true, "fallback_backend": "batched CPU FAISS", "fit_only_backend_semantic_audit": true, "full_fit_severity_recheck": true, "preferred_backend": "single-GPU CUDA FAISS if semantic audit matches CPU", "severity_gap_tolerance": 5e-06, "validation_outcomes_must_not_be_used_for_retuning": true}, "fidelity_contrasts": {"fit_severity_match": {"relative_mismatch": 0.04533, "representation_gap_ndcg10": 0.249338, "rule": "Comparator is inherited unchanged from ARC-v0.25. Backend audit may stop execution on mismatch but must not retune nprobe.", "search_gap_ndcg10": 0.238035}, "representation": {"high": "IVF-SQ8 nlist=4096 nprobe=64", "low": "IVF-PQ32 nlist=4096 m=32 nbits=8 nprobe=64"}, "search_effort": {"high": "IVF-SQ8 nlist=4096 nprobe=64", "low": "IVF-SQ8 nlist=4096 nprobe=8"}}, "frozen_source": {"v018_run": "/content/drive/MyDrive/rag-pq-checkpoints/arc-v0/cross-encoder-fever-replication-v018/20260819-015645", "v025_protocol_sha256": "62351de5e3d2993129fd0b4f67e3fd49e06b97059b050cd800645d2d6c467e24", "v025_run": "/content/drive/MyDrive/rag-pq-checkpoints/arc-v0/fever-e5-severity-matched-mechanism-v025/20260824-095157", "validation_retuning_allowed": false}, "long_horizon": {"analysis_horizons": [4, 8, 12, 16, 24, 32, 40, 50], "max_feedback_updates": 50, "policies": [{"alpha": 0.1, "config_key": "mean-k20-a0p1-tnone", "family": "mean", "k": 20, "temperature": null}, {"alpha": 0.1, "config_key": "softmax-k20-a0p1-t0p1", "family": "softmax", "k": 20, "temperature": 0.1}, {"alpha": 0.3, "config_key": "mean-k20-a0p3-tnone", "family": "mean", "k": 20, "temperature": null}, {"alpha": 0.3, "config_key": "softmax-k20-a0p3-t0p1", "family": "softmax", "k": 20, "temperature": 0.1}, {"alpha": 0.5, "config_key": "mean-k20-a0p5-tnone", "family": "mean", "k": 20, "temperature": null}, {"alpha": 0.5, "config_key": "softmax-k20-a0p5-t0p1", "family": "softmax", "k": 20, "temperature": 0.1}, {"alpha": 0.7, "config_key": "mean-k20-a0p7-tnone", "family": "mean", "k": 20, "temperature": null}, {"alpha": 0.7, "config_key": "softmax-k20-a0p7-t0p1", "family": "softmax", "k": 20, "temperature": 0.1}], "policy_count": 8, "policy_selection_rule": "Structural subset copied from ARC-v0.28: for each alpha, mean-k20 and softmax-k20-tau0.1. Selection is outcome-independent."}, "n_documents": 5416568, "operators": {"anchored": "q_(t+1)=normalize((1-alpha)*q0 + alpha*F_t)", "recursive": "q_(t+1)=normalize((1-alpha)*q_t + alpha*F_t)"}, "primary_hypothesis": {"direction": "negative", "estimand": "representation_H3abs - search_effort_H3abs", "horizon": 50, "operator": "recursive", "retention_rule": "Negative, null, positive, saturation, non-replication, or a different transition horizon must be retained unchanged; no comparator, policy, horizon, endpoint, or split retuning.", "success_criterion": "10,000-replicate paired query-bootstrap 95% CI lies strictly below zero"}, "retrieval": {"metric": "inner_product", "top_retrieve": 100, "utility_k": 10}, "scientific_goal": "Test whether the NQ-GTE recursive long-horizon representation-minus-search H3abs reversal replicates on FEVER-E5 under the already frozen v0.25 severity match.", "secondary_prespecified": ["recursive paired contrast at H=4,8,12,16,24,32,40", "anchored paired contrast at H=4,8,12,16,24,32,40,50", "mean-only and softmax-only paired contrasts", "MRR@10 and Recall@10 H3abs paired contrasts", "H=8 to H=50 raw absolute-gap change", "final-minus-initial absolute-gap change", "operator interaction at H=50: recursive minus anchored paired mechanism contrast"], "split": {"fit_membership_sha256": "85c01de943636a080abe10cb18cf5538b705b89f6f042a74c92bac366d89612c", "n_fit": 3350, "n_validation": 3316, "source": "ARC-v0.13 authoritative FIT/validation split reused by ARC-v0.18/v0.25", "validation_membership_sha256": "5e7bd8e3e5e3f0120b1e93726a19183285624c405ef68b34005c8766a9568b42"}, "statistics": {"independent_unit": "query", "paired_query_bootstrap_reps": 10000, "policy_aggregation": "average endpoint within query over the frozen 8-policy subset before mechanism contrast", "seed": 20260826}, "status": "ARC_V029_FEVER_E5_H50_OPERATOR_HORIZON_PROTOCOL_FROZEN_BEFORE_NEW_LONG_HORIZON_VALIDATION_OUTCOMES", "study_id": "ARC-v0.29", "test_accessed": false, "test_relevance_accessed": false}'
EXPECTED_V029_PROTOCOL_SHA = "c24ba59c58a37a87f0823afff0854085c36cdbc0d3c8da7800873f246c9b6cf8"

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    if drive is None:
        raise RuntimeError("Google Drive is not mounted and google.colab.drive is unavailable.")
    drive.mount("/content/drive")

ARC_ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v0"
V018 = ARC_ROOT / "cross-encoder-fever-replication-v018" / "20260819-015645"
V025 = ARC_ROOT / "fever-e5-severity-matched-mechanism-v025" / "20260824-095157"
SPLIT_PATH = ARC_ROOT / "fever-boundary-external-replication-v013" / "20260817-151852" / "v013_boundary_query_split.csv"

OUT_ROOT = ARC_ROOT / "fever-e5-operator-horizon-replication-v029"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Freeze one canonical protocol at the study root before reading new long-horizon validation outcomes.
PROTOCOL_PATH = OUT_ROOT / "V029_FROZEN_PROTOCOL.json"
canonical = json.dumps(json.loads(V029_PROTOCOL_JSON), indent=2, sort_keys=True) + "\n"
got_sha = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
assert got_sha == EXPECTED_V029_PROTOCOL_SHA, (got_sha, EXPECTED_V029_PROTOCOL_SHA)

if PROTOCOL_PATH.exists():
    existing_sha = hashlib.sha256(PROTOCOL_PATH.read_bytes()).hexdigest()
    assert existing_sha == EXPECTED_V029_PROTOCOL_SHA, (
        "Existing v0.29 protocol differs from the frozen protocol.",
        existing_sha, EXPECTED_V029_PROTOCOL_SHA
    )
else:
    PROTOCOL_PATH.write_text(canonical, encoding="utf-8")

(OUT_ROOT / "V029_PROTOCOL_SHA256.txt").write_text(
    f"{EXPECTED_V029_PROTOCOL_SHA}  {PROTOCOL_PATH.name}\n", encoding="utf-8"
)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = OUT_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

faiss.omp_set_num_threads(os.cpu_count() or 1)
print("FAISS:", getattr(faiss, "__version__", "unknown"))
print("GPU count:", getattr(faiss, "get_num_gpus", lambda: 0)())
print("Protocol:", PROTOCOL_PATH)
print("Protocol SHA:", EXPECTED_V029_PROTOCOL_SHA)
print("Run:", OUT)
print("V0.29 PROTOCOL FREEZE — PASS")

In [ ]:
# Cell 2 — Provenance, split, and source-artifact audit
def sha256_file(path, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def membership_sha(ids):
    return hashlib.sha256(
        "\n".join(sorted(map(str, ids))).encode("utf-8")
    ).hexdigest()

V018_PROTOCOL = V018 / "v018_cross_encoder_protocol.json"
V025_PROTOCOL = V025 / "v025_frozen_severity_match_protocol.json"
QUERY_IDS_PATH = V018 / "dev_query_ids.txt"
QUERY_EMB_PATH = V018 / "dev_query_embeddings.float32.npy"
CORPUS_MEMMAP_PATH = V018 / "corpus_embeddings.float16.memmap"
PQ_PATH = V018 / "fever-e5-small-v2-ivfpq-nlist4096-m32-nbits8.faiss"
SQ_PATH = V018 / "fever-e5-small-v2-ivfsq8-nlist4096.faiss"
SEALED_QRELS = DRIVE_ROOT / "hc-rars-fever-5m-untouched-confirmation-v1" / "stage2" / "dev_qrels_rows.csv"

required = [
    V018_PROTOCOL, V025_PROTOCOL, QUERY_IDS_PATH, QUERY_EMB_PATH,
    CORPUS_MEMMAP_PATH, PQ_PATH, SQ_PATH, SPLIT_PATH, SEALED_QRELS
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing required frozen artifacts:\n" + "\n".join(missing))

assert sha256_file(V025_PROTOCOL) == EXPECTED_V025_PROTOCOL_SHA
v018_protocol = json.loads(V018_PROTOCOL.read_text())
v025_protocol = json.loads(V025_PROTOCOL.read_text())

assert v018_protocol["encoder"] == "intfloat/e5-small-v2"
assert int(v018_protocol["dimension"]) == DIM
assert int(v018_protocol["retrievers"]["PQ32"]["nprobe"]) == HIGH_NPROBE
assert int(v018_protocol["retrievers"]["SQ8"]["nprobe"]) == HIGH_NPROBE
assert int(v025_protocol["selected_search_effort_low_nprobe"]) == MATCHED_NPROBE
assert v025_protocol["calibration_split"] == "FIT only"
assert v025_protocol["post_selection_retuning_allowed"] is False

split = pd.read_csv(SPLIT_PATH)
split["query_id"] = split["query_id"].astype(str)
FIT_IDS = split.loc[split["split"].eq("fit"), "query_id"].tolist()
VAL_IDS = split.loc[split["split"].eq("validation"), "query_id"].tolist()

assert len(FIT_IDS) == N_FIT
assert len(VAL_IDS) == N_VAL
assert membership_sha(FIT_IDS) == EXPECTED_FIT_SHA
assert membership_sha(VAL_IDS) == EXPECTED_VAL_SHA
assert set(FIT_IDS).isdisjoint(VAL_IDS)

expected_bytes = N_DOCS * DIM * np.dtype(np.float16).itemsize
assert CORPUS_MEMMAP_PATH.stat().st_size == expected_bytes

lineage = {
    "v018_protocol_sha256": sha256_file(V018_PROTOCOL),
    "v025_protocol_sha256": sha256_file(V025_PROTOCOL),
    "split_file_sha256": sha256_file(SPLIT_PATH),
    "pq_index_sha256": sha256_file(PQ_PATH),
    "sq_index_sha256": sha256_file(SQ_PATH),
    "query_embeddings_sha256": sha256_file(QUERY_EMB_PATH),
    "qrels_sha256": sha256_file(SEALED_QRELS),
    "fit_membership_sha256": membership_sha(FIT_IDS),
    "validation_membership_sha256": membership_sha(VAL_IDS),
}
(OUT / "v029_runtime_lineage.json").write_text(json.dumps(lineage, indent=2, sort_keys=True))

print("FIT/VAL:", len(FIT_IDS), len(VAL_IDS))
print("v0.25 protocol SHA:", lineage["v025_protocol_sha256"])
print("SOURCE LINEAGE — PASS")

In [ ]:
# Cell 3 — Load queries, row-space qrels, and localize the corpus/indexes
DEV_QUERY_IDS = QUERY_IDS_PATH.read_text().splitlines()
query_mm = np.load(QUERY_EMB_PATH, mmap_mode="r")
assert query_mm.shape == (6666, DIM)
qid_to_query_row = {str(qid): i for i, qid in enumerate(DEV_QUERY_IDS)}

assert all(q in qid_to_query_row for q in FIT_IDS)
assert all(q in qid_to_query_row for q in VAL_IDS)

qr = pd.read_csv(SEALED_QRELS)
assert {"query-id", "corpus-row", "score"}.issubset(qr.columns)
qr["query-id"] = qr["query-id"].astype(str)
qr["corpus-row"] = pd.to_numeric(qr["corpus-row"], errors="raise").astype(np.int64)
qr["score"] = pd.to_numeric(qr["score"], errors="coerce")
qr = qr[qr["score"] > 0].copy()

QRELS = defaultdict(list)
for qid, row in zip(qr["query-id"], qr["corpus-row"]):
    QRELS[str(qid)].append(int(row))

assert all(len(QRELS[q]) > 0 for q in FIT_IDS + VAL_IDS)

def row_normalize(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(n, eps)

FIT_Q = row_normalize(np.asarray(query_mm[[qid_to_query_row[q] for q in FIT_IDS]], dtype=np.float32))
VAL_Q = row_normalize(np.asarray(query_mm[[qid_to_query_row[q] for q in VAL_IDS]], dtype=np.float32))

LOCAL_ROOT = Path("/content/arc-v029-local")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

def local_copy(src):
    dst = LOCAL_ROOT / src.name
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        print("Localizing:", src.name)
        tmp = dst.with_suffix(dst.suffix + ".tmp")
        shutil.copy2(src, tmp)
        os.replace(tmp, dst)
    return dst

if LOCALIZE_TO_SSD:
    local_corpus = local_copy(CORPUS_MEMMAP_PATH)
    local_pq = local_copy(PQ_PATH)
    local_sq = local_copy(SQ_PATH)
else:
    local_corpus, local_pq, local_sq = CORPUS_MEMMAP_PATH, PQ_PATH, SQ_PATH

corpus = np.memmap(
    local_corpus, dtype=np.float16, mode="r", shape=(N_DOCS, DIM)
)

print("FIT_Q:", FIT_Q.shape, "VAL_Q:", VAL_Q.shape)
print("Corpus memmap:", corpus.shape, corpus.dtype)
print("DATA LOAD — PASS")

In [ ]:
# Cell 4 — Vectorized relevance metrics
def make_rel_matrix(qids):
    max_rel = max(len(QRELS[q]) for q in qids)
    mat = np.full((len(qids), max_rel), -1, dtype=np.int64)
    counts = np.empty(len(qids), dtype=np.int32)
    for i, q in enumerate(qids):
        rel = np.asarray(sorted(set(QRELS[q])), dtype=np.int64)
        mat[i, :len(rel)] = rel
        counts[i] = len(rel)
    return mat, counts

FIT_REL, FIT_REL_COUNT = make_rel_matrix(FIT_IDS)
VAL_REL, VAL_REL_COUNT = make_rel_matrix(VAL_IDS)

DISCOUNTS = 1.0 / np.log2(np.arange(2, UTILITY_K + 2))
IDCG_CACHE = np.zeros(UTILITY_K + 1, dtype=np.float64)
for m in range(1, UTILITY_K + 1):
    IDCG_CACHE[m] = DISCOUNTS[:m].sum()

def relevance_hits(top_ids, rel_matrix):
    ids = np.asarray(top_ids[:, :UTILITY_K], dtype=np.int64)
    return (ids[:, :, None] == rel_matrix[:, None, :]).any(axis=2)

def metrics_from_ids(top_ids, rel_matrix, rel_counts):
    hits = relevance_hits(top_ids, rel_matrix)
    dcg = (hits * DISCOUNTS[None, :]).sum(axis=1)
    ideal_m = np.minimum(rel_counts, UTILITY_K)
    idcg = IDCG_CACHE[ideal_m]
    ndcg = np.divide(dcg, idcg, out=np.zeros_like(dcg), where=idcg > 0)

    any_hit = hits.any(axis=1)
    first = np.argmax(hits, axis=1)
    mrr = np.where(any_hit, 1.0 / (first + 1), 0.0)

    recall = hits.sum(axis=1) / np.maximum(rel_counts, 1)
    return ndcg.astype(np.float32), mrr.astype(np.float32), recall.astype(np.float32)

print("Max FIT rels/query:", FIT_REL.shape[1])
print("Max VAL rels/query:", VAL_REL.shape[1])
print("METRIC HELPERS — PASS")

In [ ]:
# Cell 5 — Load CPU indexes and create audited GPU copies if supported
print("Loading CPU PQ32...")
pq_cpu = faiss.read_index(str(local_pq))
print("Loading CPU SQ8...")
sq_cpu = faiss.read_index(str(local_sq))
assert pq_cpu.ntotal == N_DOCS and sq_cpu.ntotal == N_DOCS

pq_cpu.nprobe = HIGH_NPROBE
sq_cpu.nprobe = HIGH_NPROBE

gpu_resources = []
pq_gpu = sq_gpu = None
GPU_AVAILABLE = False

if getattr(faiss, "get_num_gpus", lambda: 0)() > 0 and hasattr(faiss, "StandardGpuResources"):
    try:
        res_pq = faiss.StandardGpuResources()
        res_sq = faiss.StandardGpuResources()
        pq_gpu = faiss.index_cpu_to_gpu(res_pq, 0, pq_cpu)
        sq_gpu = faiss.index_cpu_to_gpu(res_sq, 0, sq_cpu)
        gpu_resources = [res_pq, res_sq]
        GPU_AVAILABLE = True
        print("GPU index transfer — PASS")
    except Exception as e:
        print("GPU transfer unavailable; falling back to CPU:", repr(e))
        pq_gpu = sq_gpu = None
        GPU_AVAILABLE = False

def search_index(index, queries, nprobe, batch=SEARCH_BATCH):
    index.nprobe = int(nprobe)
    all_s, all_i = [], []
    for st in range(0, len(queries), batch):
        q = np.ascontiguousarray(queries[st:st+batch], dtype=np.float32)
        s, i = index.search(q, TOP_RETRIEVE)
        all_s.append(s)
        all_i.append(i)
    return np.vstack(all_s), np.vstack(all_i)

print("GPU_AVAILABLE:", GPU_AVAILABLE)
print("INDEX LOAD — PASS")

In [ ]:
# Cell 6 — FIT-only backend semantic audit and frozen severity recheck
AUDIT_N = 256
audit_idx = np.arange(AUDIT_N)
audit_q = FIT_Q[audit_idx]
audit_rel = FIT_REL[audit_idx]
audit_count = FIT_REL_COUNT[audit_idx]

def condition_searches(pq_index, sq_index, q):
    s_pq, i_pq = search_index(pq_index, q, HIGH_NPROBE)
    s_sq64, i_sq64 = search_index(sq_index, q, HIGH_NPROBE)
    s_sq8, i_sq8 = search_index(sq_index, q, MATCHED_NPROBE)
    return {
        "pq64": (s_pq, i_pq),
        "sq64": (s_sq64, i_sq64),
        "sq8": (s_sq8, i_sq8),
    }

cpu_audit = condition_searches(pq_cpu, sq_cpu, audit_q)
cpu_metric = {}
for name, (_, ids) in cpu_audit.items():
    cpu_metric[name] = metrics_from_ids(ids, audit_rel, audit_count)[0]

backend_audit = {
    "n_fit_audit_queries": AUDIT_N,
    "gpu_available": bool(GPU_AVAILABLE),
    "max_abs_ndcg_diff": {},
}

if GPU_AVAILABLE:
    gpu_audit = condition_searches(pq_gpu, sq_gpu, audit_q)
    for name, (_, ids) in gpu_audit.items():
        g = metrics_from_ids(ids, audit_rel, audit_count)[0]
        delta = float(np.max(np.abs(g - cpu_metric[name])))
        backend_audit["max_abs_ndcg_diff"][name] = delta
        if delta > 1e-12:
            raise RuntimeError(
                f"CPU/GPU semantic mismatch for {name}: max |ΔnDCG|={delta}. "
                "STOP. Do not retune the frozen comparator."
            )
    ACTIVE_PQ, ACTIVE_SQ = pq_gpu, sq_gpu
    ACTIVE_BACKEND = "gpu"
else:
    ACTIVE_PQ, ACTIVE_SQ = pq_cpu, sq_cpu
    ACTIVE_BACKEND = "cpu"

# Recheck the original FIT severity calibration on all 3,350 FIT queries.
_, fit_pq_ids = search_index(ACTIVE_PQ, FIT_Q, HIGH_NPROBE)
_, fit_sq64_ids = search_index(ACTIVE_SQ, FIT_Q, HIGH_NPROBE)
_, fit_sq8_ids = search_index(ACTIVE_SQ, FIT_Q, MATCHED_NPROBE)

fit_pq_ndcg = metrics_from_ids(fit_pq_ids, FIT_REL, FIT_REL_COUNT)[0]
fit_sq64_ndcg = metrics_from_ids(fit_sq64_ids, FIT_REL, FIT_REL_COUNT)[0]
fit_sq8_ndcg = metrics_from_ids(fit_sq8_ids, FIT_REL, FIT_REL_COUNT)[0]

rep_gap = float(fit_sq64_ndcg.mean() - fit_pq_ndcg.mean())
search_gap = float(fit_sq64_ndcg.mean() - fit_sq8_ndcg.mean())

expected_rep = float(v025_protocol["fit_representation_gap"])
expected_search = float(v025_protocol["fit_matched_search_gap"])
tol = 5e-6

backend_audit.update({
    "active_backend": ACTIVE_BACKEND,
    "fit_representation_gap": rep_gap,
    "fit_search_gap": search_gap,
    "expected_representation_gap": expected_rep,
    "expected_search_gap": expected_search,
    "gap_tolerance": tol,
})

if abs(rep_gap - expected_rep) > tol or abs(search_gap - expected_search) > tol:
    raise RuntimeError(
        "Frozen FIT severity calibration was not reproduced within tolerance. "
        f"rep {rep_gap} vs {expected_rep}; search {search_gap} vs {expected_search}. "
        "STOP. Do not choose a new nprobe."
    )

(OUT / "v029_fit_backend_semantic_audit.json").write_text(
    json.dumps(backend_audit, indent=2, sort_keys=True)
)

print(json.dumps(backend_audit, indent=2))
print("FIT-ONLY BACKEND + SEVERITY AUDIT — PASS")
print("Validation long-horizon trajectories remain untouched.")

In [ ]:
# Cell 7 — Frozen 8-policy long-horizon subset and trajectory helpers
POLICIES = []
for alpha in [0.1, 0.3, 0.5, 0.7]:
    a = str(alpha).replace(".", "p")
    POLICIES.append({
        "config_key": f"mean-k20-a{a}-tnone",
        "family": "mean",
        "alpha": float(alpha),
        "k": 20,
        "temperature": np.nan,
    })
    POLICIES.append({
        "config_key": f"softmax-k20-a{a}-t0p1",
        "family": "softmax",
        "alpha": float(alpha),
        "k": 20,
        "temperature": 0.1,
    })

assert len(POLICIES) == 8
pd.DataFrame(POLICIES).to_csv(OUT / "v029_frozen_h50_policy_subset.csv", index=False)

def feedback_batch(scores, ids, policy):
    k = int(policy["k"])
    n = len(ids)
    out = np.empty((n, DIM), dtype=np.float32)
    for st in range(0, n, FEEDBACK_BATCH):
        en = min(st + FEEDBACK_BATCH, n)
        rows = np.asarray(ids[st:en, :k], dtype=np.int64)
        if (rows < 0).any():
            raise RuntimeError("FAISS returned invalid document IDs in the feedback top-k.")

        docs = np.asarray(corpus[rows.reshape(-1)], dtype=np.float32).reshape(en-st, k, DIM)
        docs /= np.maximum(np.linalg.norm(docs, axis=2, keepdims=True), 1e-12)

        if policy["family"] == "mean":
            f = docs.mean(axis=1)
        else:
            tau = float(policy["temperature"])
            z = np.asarray(scores[st:en, :k], dtype=np.float64) / tau
            z -= z.max(axis=1, keepdims=True)
            w = np.exp(z)
            w /= w.sum(axis=1, keepdims=True)
            f = (docs * w[:, :, None]).sum(axis=1)

        f /= np.maximum(np.linalg.norm(f, axis=1, keepdims=True), 1e-12)
        out[st:en] = f.astype(np.float32, copy=False)
    return out

def update_batch(q0, qcur, feedback, alpha, operator):
    base = q0 if operator == "anchored" else qcur
    nxt = (1.0 - float(alpha)) * base + float(alpha) * feedback
    return row_normalize(nxt)

def prefix_slope(y, H):
    y = np.asarray(y[:, :H+1], dtype=np.float64)
    x = np.arange(H + 1, dtype=np.float64)
    xc = x - x.mean()
    return (y @ xc) / float(np.dot(xc, xc))

print(pd.DataFrame(POLICIES))
print("LONG-HORIZON HELPERS — PASS")

In [ ]:
# Cell 8 — One mechanism/operator/policy run with atomic checkpointing
CHECKPOINT_DIR = OUT / "trajectory-checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def get_pair(mechanism):
    if mechanism == "representation":
        return (ACTIVE_PQ, HIGH_NPROBE), (ACTIVE_SQ, HIGH_NPROBE)
    if mechanism == "search_effort":
        return (ACTIVE_SQ, MATCHED_NPROBE), (ACTIVE_SQ, HIGH_NPROBE)
    raise ValueError(mechanism)

def run_one(mechanism, operator, policy):
    key = f"{mechanism}__{operator}__{policy['config_key']}"
    cp = CHECKPOINT_DIR / f"{key}.npz"
    if cp.exists():
        print("skip:", cp.name)
        return cp

    (idxL, npL), (idxH, npH) = get_pair(mechanism)
    q0 = VAL_Q.copy()
    qL = q0.copy()
    qH = q0.copy()
    n = len(VAL_IDS)

    state_dist = np.empty((n, MAX_H + 1), dtype=np.float32)
    ndcg_L = np.empty_like(state_dist)
    ndcg_H = np.empty_like(state_dist)
    mrr_L = np.empty_like(state_dist)
    mrr_H = np.empty_like(state_dist)
    recall_L = np.empty_like(state_dist)
    recall_H = np.empty_like(state_dist)

    t0 = time.perf_counter()
    for t in range(MAX_H + 1):
        sL, iL = search_index(idxL, qL, npL)
        sH, iH = search_index(idxH, qH, npH)

        state_dist[:, t] = 1.0 - np.sum(qL * qH, axis=1)
        ndcg_L[:, t], mrr_L[:, t], recall_L[:, t] = metrics_from_ids(iL, VAL_REL, VAL_REL_COUNT)
        ndcg_H[:, t], mrr_H[:, t], recall_H[:, t] = metrics_from_ids(iH, VAL_REL, VAL_REL_COUNT)

        if t == MAX_H:
            break

        fL = feedback_batch(sL, iL, policy)
        fH = feedback_batch(sH, iH, policy)
        qL = update_batch(q0, qL, fL, policy["alpha"], operator)
        qH = update_batch(q0, qH, fH, policy["alpha"], operator)

        if t in {7, 15, 31, 39}:
            print(f"  {key}: completed update {t+1}/{MAX_H}")

    tmp = cp.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        np.savez_compressed(
            f,
            query_ids=np.asarray(VAL_IDS),
            state_dist=state_dist,
            ndcg_L=ndcg_L, ndcg_H=ndcg_H,
            mrr_L=mrr_L, mrr_H=mrr_H,
            recall_L=recall_L, recall_H=recall_H,
            mechanism=np.asarray(mechanism),
            operator=np.asarray(operator),
            config_key=np.asarray(policy["config_key"]),
            family=np.asarray(policy["family"]),
            alpha=np.asarray(policy["alpha"], dtype=np.float32),
            temperature=np.asarray(policy["temperature"], dtype=np.float32),
        )
    os.replace(tmp, cp)
    print(f"wrote {cp.name} | {time.perf_counter()-t0:.1f}s")
    return cp

print("CHECKPOINT DIR:", CHECKPOINT_DIR)
print("Expected full long-horizon query-round records:",
      f"{N_VAL * 8 * 2 * 2 * (MAX_H + 1):,}")

## Validation gate

**Do not run the next cell until Cells 1–8 pass.**

After this point, new FEVER-E5 long-horizon validation outcomes are being generated. If a result is null, positive, negative, crosses earlier/later, or never crosses, **do not alter the frozen protocol**.

In [ ]:
# Cell 9 — Execute the frozen 32-run long-horizon validation matrix
MECHANISMS = ["representation", "search_effort"]
OPERATORS = ["anchored", "recursive"]

all_checkpoints = []
for mechanism in MECHANISMS:
    for operator in OPERATORS:
        for policy in POLICIES:
            all_checkpoints.append(run_one(mechanism, operator, policy))

assert len(all_checkpoints) == 32
assert all(Path(p).is_file() for p in all_checkpoints)
print("LONG-HORIZON VALIDATION MATRIX — COMPLETE")

In [ ]:
# Cell 10 — Convert checkpoints to per-query prefix endpoints
endpoint_rows = []

for cp in sorted(CHECKPOINT_DIR.glob("*.npz")):
    z = np.load(cp, allow_pickle=False)
    qids = z["query_ids"].astype(str)
    mechanism = str(z["mechanism"])
    operator = str(z["operator"])
    config_key = str(z["config_key"])
    family = str(z["family"])
    alpha = float(z["alpha"])
    temperature = float(z["temperature"])

    state = z["state_dist"]
    ndL, ndH = z["ndcg_L"], z["ndcg_H"]
    mrL, mrH = z["mrr_L"], z["mrr_H"]
    rcL, rcH = z["recall_L"], z["recall_H"]

    abs_nd = np.abs(ndH - ndL)
    signed_nd = ndH - ndL
    abs_mr = np.abs(mrH - mrL)
    abs_rc = np.abs(rcH - rcL)

    for H in HORIZONS:
        frame = pd.DataFrame({
            "query_id": qids,
            "mechanism": mechanism,
            "operator": operator,
            "config_key": config_key,
            "family": family,
            "alpha": alpha,
            "temperature": temperature,
            "H": H,
            "H1_slope": prefix_slope(state, H),
            "H3_abs_slope": prefix_slope(abs_nd, H),
            "H3_signed_slope": prefix_slope(signed_nd, H),
            "MRR_H3_abs_slope": prefix_slope(abs_mr, H),
            "Recall_H3_abs_slope": prefix_slope(abs_rc, H),
            "abs_gap_final_minus_initial": abs_nd[:, H] - abs_nd[:, 0],
            "abs_gap_at_H": abs_nd[:, H],
        })
        endpoint_rows.append(frame)

endpoints = pd.concat(endpoint_rows, ignore_index=True)
expected = N_VAL * 32 * len(HORIZONS)
assert len(endpoints) == expected, (len(endpoints), expected)

ENDPOINT_PATH = OUT / "v029_h50_endpoints.parquet"
endpoints.to_parquet(ENDPOINT_PATH, index=False)
print("Endpoint rows:", f"{len(endpoints):,}")
print("ENDPOINT RECOMPUTE — PASS")

In [ ]:
# Cell 11 — Frozen query-level paired bootstrap at every operator/horizon
def paired_bootstrap(x, seed, reps=BOOTSTRAP_REPS):
    x = np.asarray(x, dtype=np.float64)
    assert np.isfinite(x).all()
    n = len(x)
    rng = np.random.default_rng(seed)
    boots = np.empty(reps, dtype=np.float64)
    chunk = 250
    done = 0
    while done < reps:
        b = min(chunk, reps - done)
        idx = rng.integers(0, n, size=(b, n), dtype=np.int32)
        boots[done:done+b] = x[idx].mean(axis=1)
        done += b
    return {
        "mean": float(x.mean()),
        "ci95_low": float(np.quantile(boots, 0.025)),
        "ci95_high": float(np.quantile(boots, 0.975)),
        "n_queries": int(n),
    }

def query_policy_average(df, measure, families=None):
    x = df
    if families is not None:
        x = x[x["family"].isin(families)]
    return x.groupby("query_id", as_index=True)[measure].mean().sort_index()

summary = []
measures = ["H3_abs_slope", "MRR_H3_abs_slope", "Recall_H3_abs_slope"]

for operator in OPERATORS:
    for H in HORIZONS:
        sub = endpoints[(endpoints["operator"] == operator) & (endpoints["H"] == H)]
        for measure in measures:
            for family_label, fams in [
                ("all8_equal_family", None),
                ("mean_only", ["mean"]),
                ("softmax_only", ["softmax"]),
            ]:
                rep = query_policy_average(
                    sub[sub["mechanism"] == "representation"], measure, fams
                )
                sea = query_policy_average(
                    sub[sub["mechanism"] == "search_effort"], measure, fams
                )
                common = rep.index.intersection(sea.index)
                assert len(common) == N_VAL
                diff = (rep.loc[common] - sea.loc[common]).to_numpy()

                seed = (
                    SEED * 1000
                    + (1 if operator == "recursive" else 0) * 100
                    + H
                    + {"H3_abs_slope": 0, "MRR_H3_abs_slope": 10, "Recall_H3_abs_slope": 20}[measure]
                    + {"all8_equal_family": 0, "mean_only": 30, "softmax_only": 60}[family_label]
                )
                st = paired_bootstrap(diff, seed)
                summary.append({
                    "operator": operator,
                    "H": H,
                    "measure": measure,
                    "family_estimand": family_label,
                    "estimand": "representation_minus_search_effort",
                    **st,
                })

summary_df = pd.DataFrame(summary)
SUMMARY_PATH = OUT / "v029_horizon_paired_query_bootstrap.csv"
summary_df.to_csv(SUMMARY_PATH, index=False)

primary = summary_df[
    (summary_df["operator"] == "recursive")
    & (summary_df["H"] == 50)
    & (summary_df["measure"] == "H3_abs_slope")
    & (summary_df["family_estimand"] == "all8_equal_family")
].iloc[0]

display(summary_df[
    (summary_df["measure"] == "H3_abs_slope")
    & (summary_df["family_estimand"] == "all8_equal_family")
].sort_values(["operator", "H"]))

print("\nPRIMARY recursive H=50:")
print(primary.to_dict())

In [ ]:
# Cell 12 — Prespecified H=8→H=50 late-gap and operator-interaction diagnostics
diag_rows = []

for operator in OPERATORS:
    for mechanism in MECHANISMS:
        sub8 = endpoints[
            (endpoints["operator"] == operator)
            & (endpoints["mechanism"] == mechanism)
            & (endpoints["H"] == 8)
        ]
        sub50 = endpoints[
            (endpoints["operator"] == operator)
            & (endpoints["mechanism"] == mechanism)
            & (endpoints["H"] == 50)
        ]

        q8 = sub8.groupby("query_id")["abs_gap_at_H"].mean().sort_index()
        q50 = sub50.groupby("query_id")["abs_gap_at_H"].mean().sort_index()
        change = (q50 - q8).to_numpy()
        st = paired_bootstrap(change, SEED + 29000 + (1 if operator=="recursive" else 0) * 100 + (1 if mechanism=="representation" else 2))
        diag_rows.append({
            "diagnostic": "raw_abs_gap_H50_minus_H8",
            "operator": operator,
            "mechanism": mechanism,
            **st,
        })

# Mechanism contrast of late-gap change under recursive feedback.
rec8 = endpoints[(endpoints["operator"]=="recursive") & (endpoints["H"]==8)]
rec50 = endpoints[(endpoints["operator"]=="recursive") & (endpoints["H"]==50)]

def qgap(df, mech):
    return df[df["mechanism"]==mech].groupby("query_id")["abs_gap_at_H"].mean().sort_index()

rep_change = qgap(rec50, "representation") - qgap(rec8, "representation")
sea_change = qgap(rec50, "search_effort") - qgap(rec8, "search_effort")
paired_late = (rep_change - sea_change).to_numpy()
st = paired_bootstrap(paired_late, SEED + 29901)
diag_rows.append({
    "diagnostic": "recursive_paired_mechanism_raw_gap_change_H50_minus_H8",
    "operator": "recursive",
    "mechanism": "representation_minus_search_effort",
    **st,
})

# Operator interaction at H=50 on H3abs mechanism contrast.
def mechanism_diff_by_operator(op):
    sub = endpoints[(endpoints["operator"]==op) & (endpoints["H"]==50)]
    rep = sub[sub["mechanism"]=="representation"].groupby("query_id")["H3_abs_slope"].mean().sort_index()
    sea = sub[sub["mechanism"]=="search_effort"].groupby("query_id")["H3_abs_slope"].mean().sort_index()
    return rep - sea

interaction = (mechanism_diff_by_operator("recursive") - mechanism_diff_by_operator("anchored")).to_numpy()
st = paired_bootstrap(interaction, SEED + 29902)
diag_rows.append({
    "diagnostic": "H50_recursive_minus_anchored_mechanism_contrast",
    "operator": "interaction",
    "mechanism": "representation_minus_search_effort",
    **st,
})

diag_df = pd.DataFrame(diag_rows)
DIAG_PATH = OUT / "v029_prespecified_late_horizon_diagnostics.csv"
diag_df.to_csv(DIAG_PATH, index=False)
display(diag_df)

In [ ]:
# Cell 13 — Frozen primary gate, outcome classification, and artifact hashes
primary_mean = float(primary["mean"])
primary_lo = float(primary["ci95_low"])
primary_hi = float(primary["ci95_high"])

if primary_hi < 0:
    primary_class = "NEGATIVE_REPLICATION_SUCCESS"
elif primary_lo > 0:
    primary_class = "POSITIVE_NONREPLICATION"
else:
    primary_class = "UNRESOLVED_NULL_OR_MIXED"

gate = {
    "study_id": "ARC-v0.29",
    "status": "FEVER_E5_LONG_HORIZON_VALIDATION_ANALYZED",
    "protocol_sha256": EXPECTED_V029_PROTOCOL_SHA,
    "classification": (
        "prospective FEVER-E5 operator/horizon replication/extension; "
        "not a pristine independent test because short-horizon FEVER validation outcomes predated v0.29"
    ),
    "active_backend": ACTIVE_BACKEND,
    "n_validation_queries": N_VAL,
    "policy_count": 8,
    "horizons": HORIZONS,
    "primary": {
        "estimand": "recursive_H50_representation_minus_search_H3abs",
        "mean": primary_mean,
        "ci95": [primary_lo, primary_hi],
        "frozen_direction": "negative",
        "success_criterion": "95% paired-query bootstrap CI strictly below zero",
        "outcome_class": primary_class,
    },
    "retuning_performed": False,
    "negative_null_positive_or_nonreplication_retained": True,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
}

GATE_PATH = OUT / "v029_primary_h50_replication_gate.json"
GATE_PATH.write_text(json.dumps(gate, indent=2, sort_keys=True))

artifact_paths = [
    PROTOCOL_PATH,
    OUT / "v029_runtime_lineage.json",
    OUT / "v029_fit_backend_semantic_audit.json",
    OUT / "v029_frozen_h50_policy_subset.csv",
    ENDPOINT_PATH,
    SUMMARY_PATH,
    DIAG_PATH,
    GATE_PATH,
]
hash_rows = []
for p in artifact_paths:
    if p.exists():
        hash_rows.append({
            "file": str(p),
            "bytes": p.stat().st_size,
            "sha256": sha256_file(p),
        })

hash_df = pd.DataFrame(hash_rows)
HASH_PATH = OUT / "V029_ARTIFACT_SHA256.csv"
hash_df.to_csv(HASH_PATH, index=False)

print(json.dumps(gate, indent=2))
display(hash_df)
print("=" * 92)
print("ARC-v0.29 COMPLETE")
print("OUT:", OUT)
print("PRIMARY OUTCOME:", primary_class)
print("=" * 92)

## Send back after execution

Please send back either the executed notebook or these four files:

1. `v029_primary_h50_replication_gate.json`
2. `v029_horizon_paired_query_bootstrap.csv`
3. `v029_prespecified_late_horizon_diagnostics.csv`
4. `v029_fit_backend_semantic_audit.json`

The paper-facing decision must use the frozen primary **as written**, regardless of whether FEVER reproduces the NQ reversal.